# BasuraBin: classical vs. scratch CNN vs. pretrained CNN for household waste segregation

**Computer Vision Comparison Project, West Visayas State University**

**Task:** image classification into the three household bins required by RA 9003 (Ecological Solid Waste Management Act):
**Biodegradable** (*nabubulok*), **Recyclable**, and **Non-biodegradable / Residual** (*di-nabubulok*).

**Research question:** *For sorting household waste photos into the three RA 9003 bins, how much accuracy do learned features
(a CNN trained from scratch, and an ImageNet-pretrained CNN) gain over hand-designed features, and what does each cost in speed and size
for a real-time camera app?*

| Slot | Method | Why this method |
|---|---|---|
| A. Classical | HOG + HSV colour + uniform LBP histograms (spatial pyramid), SVM with a linear or RBF kernel (NumPy, from scratch) | hand-designed features, 0 learned features |
| B. Trained from scratch | WasteNet-R, our residual CNN (~1.6 M params, PyTorch, random init) | learned features, small, no outside knowledge |
| C. Pretrained, modern | MobileNetV3-Large (2019), ImageNet weights, fine-tuned | learned features + transfer learning, built for phones |

All three use **the same images, the same train/val/test split and the same metrics** (`splits/*.csv`).

Run every cell from top to bottom (works in Google Colab and on a local Jupyter with a GPU).

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/nylla8444/comvis-final.git"
if IN_COLAB:
    if not os.path.exists("/content/comvis-final"):
        !git clone -q {REPO_URL} /content/comvis-final
    os.chdir("/content/comvis-final")
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

QUICK = False  # True = tiny epoch counts, only for checking that the notebook runs

import json, numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from IPython.display import Image as ShowImage, Markdown, display
from common import CLASSES, METHOD_INFO, MODELS_DIR, RESULTS_DIR, SPLITS_DIR, load_json, load_rgb, load_split, plot_history, set_seed
set_seed(42)
print("working dir:", os.getcwd())
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 1. Data

Two public datasets are merged and relabelled into the three bins (no Kaggle login needed):

| Source | Images | Link | Licence |
|---|---|---|---|
| Garbage Classification (12 classes), web photos | 15,515 | https://www.kaggle.com/datasets/mostafaabla/garbage-classification | see Kaggle page |
| RealWaste, real landfill photos (Wollongong, AU) | 4,752 | https://archive.ics.uci.edu/dataset/908/realwaste (mirror: kaggle.com/datasets/joebeachcapital/realwaste) | CC BY 4.0 |

Mapping to bins: biological, food organics, vegetation → **biodegradable**; paper, cardboard, metal, plastic, glass → **recyclable**;
trash, misc. trash, textile trash, clothes, shoes → **residual**. Batteries are dropped (special/hazardous waste under RA 9003).

`prepare_data.py` removes duplicate images (difference hash), caps each bin at 2,500 images sampled evenly across source classes
(so the 5,325 clothes photos cannot dominate the residual bin), splits 70/15/15 stratified by source class (seed 42), and stores
every image as a 256×256 centre crop.

**Our own photos (optional).** Webcam/phone photos saved with `python live.py` (keys 1/2/3) go to `data/raw/local/<bin>/`
(an extra test set, never trained on) or, with `--save-to train`, to `data/raw/local_train/<bin>/` (added to the training split
for all three methods). Different objects are used in the two folders, so the local test measures new items.

In [ ]:
import download_data, prepare_data
download_data.main([])
prepare_data.main([])

In [ ]:
summary = load_json(SPLITS_DIR / "summary.json")
display(pd.DataFrame(summary["counts"]).T.assign(total=lambda d: d.sum(1)))
print("total images:", summary["total_images"], "| duplicates:", summary["duplicates"])

train_df = load_split("train")
fig, axes = plt.subplots(3, 8, figsize=(16, 6.5))
for r, c in enumerate(CLASSES):
    sub = train_df[train_df.label == c].sample(8, random_state=0)
    for ax, (_, row) in zip(axes[r], sub.iterrows()):
        ax.imshow(load_rgb(row.abs_path)); ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(row.source_class, fontsize=8)
    axes[r, 0].set_ylabel(c, fontsize=11)
fig.suptitle("Training samples per bin (title = original source class)"); plt.tight_layout(); plt.show()

## 2. Method A: Classical (HOG + colour + texture → SVM), written in NumPy

* **HOG** (Dalal & Triggs, 2005): 128×128 grey image, centred-difference gradients, 9 unsigned orientation bins with linear
  interpolation, 16×16-px cells, 2×2-cell blocks with L2-Hys normalisation → 1,764 values (shape/edges).
* **HSV histogram**: joint 8×4×4 hue/saturation/value histogram, square-root normalised → 128 values per region (colour).
* **Uniform LBP** (Ojala et al., 2002): 8 neighbours at radius 1, 58 uniform patterns + 1 → 59 values per region (texture).
* **Spatial pyramid** (Lazebnik et al., 2006): the colour and texture histograms are computed for the whole image *and* for each
  quarter, so the features also say *where* a colour or texture is → 640 + 295 values. Total: 2,699 features.
* **SVM**: multi-class hinge loss + L2, mini-batch SGD with momentum and cosine learning rate. Two kernels are tried:
  *linear*, and *RBF* approximated with 4,096 random Fourier features (Rahimi & Recht, 2007), which gives curved decision
  boundaries. Kernel, RBF width and regularisation are chosen on the validation split; the epoch with the best validation
  macro-F1 is kept.

No library is called for the main step: features, the random-feature map and the SVM optimiser are in `classical.py`.

### 2.1 Sanity check on synthetic images

In [ ]:
import classical
sanity = classical.sanity_check()
fig = classical.visualize_features(load_rgb(train_df.iloc[0].abs_path)); plt.show()

### 2.2 Training on the real data

In [ ]:
meta_a = classical.train(epochs=8 if QUICK else 60)
display(pd.DataFrame(meta_a["grid"]).round(3))
plot_history(meta_a["history"], "Classical: " + meta_a["display_name"] + " (best grid setting)"); plt.show()

### 2.3 Feature ablation: which hand-designed cue matters, and does the spatial pyramid help?

Same linear SVM setting in every row, so only the features change.

In [ ]:
ablation = classical.feature_ablation(epochs=8 if QUICK else 40)
display(pd.DataFrame(ablation))

## 3. Method B: WasteNet-R, a residual CNN trained from scratch

Input 160×160, random (He) initialisation:
* stem: `conv3×3 (32, stride 2) → BatchNorm → ReLU` (80×80)
* 4 stages of residual blocks (He et al., 2016) with 1, 2, 2, 1 blocks and 32, 64, 128, 256 channels; each new stage halves the
  resolution. A block is `conv3×3 → BN → ReLU → conv3×3 → BN`, added to its input (1×1 conv when the shape changes), then ReLU.
* global average pooling, dropout 0.3, linear layer to 3 classes.

The skip connections let the gradient reach the early layers directly, so this deeper network trains as easily as a shallow one;
the stride-2 stem keeps the compute below the first version's even with larger inputs. The first version (WasteNet: four plain
`[conv3×3 → BN → ReLU] ×2 → max-pool` blocks, 128×128 input) is still available with `python scratch_cnn.py --arch plain`.

Against **overfitting**: data augmentation (random crop/zoom, flips, ±20° rotation, colour jitter, blur), batch norm, dropout,
AdamW weight decay 5e-4, label smoothing 0.1, class-weighted loss, and early stopping on validation macro-F1 (patience 25).

In [ ]:
import scratch_cnn
from train_utils import count_macs, count_params
model_b = scratch_cnn.build_model("resnet")
print(model_b)
print(f"parameters: {count_params(model_b):.3f} M | compute: {count_macs(model_b, scratch_cnn.IMG_SIZE):.0f} M multiply-adds per image")
meta_b = scratch_cnn.train(epochs=2 if QUICK else 150)
plot_history(meta_b["history"], "WasteNet-R (scratch): training vs validation"); plt.show()

## 4. Method C: MobileNetV3-Large, pretrained on ImageNet, fine-tuned

Loaded from torchvision (`IMAGENET1K_V2` weights); the last layer is replaced by a 3-class layer. 224×224 inputs normalised
with ImageNet statistics; same splits, augmentation and loss as WasteNet-R.
Stage 1: backbone frozen, head trained for 3 epochs. Stage 2: whole network, backbone LR = head LR / 10, early stopping (patience 5).

In [ ]:
import pretrained
meta_c = pretrained.train(head_epochs=1 if QUICK else 3, epochs=1 if QUICK else 15)
ax_l, ax_m = plot_history(meta_c["history"], "MobileNetV3 (fine-tuned): training vs validation")
for ax in (ax_l, ax_m): ax.axvline(meta_c["stage1_epochs_run"] + 0.5, color="grey", linestyle=":")
plt.show()

## 5. Experiments and results (same test split for all three)

1. Main comparison: top-1 accuracy, macro-F1, confusion matrix, per-class F1
2. Efficiency: runtime per image (CPU and GPU), parameters (M), training time (min)
3. Per-source accuracy: web photos (Garbage Classification) vs real landfill photos (RealWaste)
4. Robustness: blur, darker, brighter, noise, 30° rotation (what a phone/webcam does to an image)
5. Our own local photos (`data/raw/local/<bin>/`), if provided

The table also lists **MACs** (multiply-accumulate operations per image): the compute that decides CPU speed, which does not
always follow the parameter count.

In [ ]:
import evaluate
out = evaluate.evaluate()
display(Markdown((RESULTS_DIR / "results_table.md").read_text()))

In [ ]:
res = out["results"]
display(pd.DataFrame({METHOD_INFO[k]["short"]: {c: v["per_class"][c]["f1"] for c in CLASSES} for k, v in
                      ((k, r["test"]) for k, r in res.items())}).round(1).rename_axis("per-class F1 (%)"))
display(pd.DataFrame({METHOD_INFO[k]["short"]: r["per_source_accuracy"] for k, r in res.items()}).round(1).rename_axis("accuracy (%) by source"))
if out["robustness_macro_f1"]:
    display(pd.DataFrame({METHOD_INFO[k]["short"]: v for k, v in out["robustness_macro_f1"].items()}).round(1).rename_axis("macro-F1 (%) under corruption"))
if any("local_test" in r for r in res.values()):
    display(pd.DataFrame({METHOD_INFO[k]["short"]: {"accuracy": r["local_test"]["accuracy"], "macro F1": r["local_test"]["macro_f1"]}
                          for k, r in res.items() if "local_test" in r}).round(1).rename_axis("our local photos"))

In [ ]:
for name in ["confusion_matrices.png", "training_curves.png", "robustness.png"]:
    if (RESULTS_DIR / name).exists():
        display(ShowImage(filename=str(RESULTS_DIR / name)))

In [ ]:
evaluate.prediction_grid("test", n=12); plt.show()
for k in res:
    p = RESULTS_DIR / f"misclassified_{k}.png"
    if p.exists():
        display(ShowImage(filename=str(p)))

## 6. Recommendation used by the app

In [ ]:
display(Markdown(out["recommendation"]["explanation"]))

## 7. The application

The app runs all three methods on an uploaded image, an uploaded video, or the live webcam, draws each result on the input and
names the recommended method (read from `results/results.json`). In the live modes each method's answer is averaged over the
last few frames, so the label does not flicker.

```
python app.py      # web app (image / video / live webcam / model comparison)
python live.py     # full-screen OpenCV webcam demo; keys 1/2/3 save the frame as a biodegradable/recyclable/residual photo
```